# ODIR-5K Ablation Study — Swin Transformer (EXP 4, 5, 6)

**3 thực nghiệm Swin-Tiny:**
- EXP 4: Ảnh gốc, không tiền xử lý
- EXP 5: Tiền xử lý (ROI+Ben Graham+CLAHE), không MixUp/CutMix
- EXP 6: Tiền xử lý + MixUp + CutMix

In [ ]:
# -- CELL 0: Khám phá cấu trúc thư mục Kaggle --------------
import os

print('📂 Cấu trúc /kaggle/input (3 cấp):')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}📁 {os.path.basename(root)}/')
    if depth >= 3:
        dirs.clear()
        continue
    for f in sorted(files)[:5]:
        print(f'{indent}  📄 {f}')


In [ ]:
# -- CELL 1: Cài thư viện ----------------------------------
!pip install timm albumentations pyyaml scikit-learn -q
print('✅ Libraries installed')

In [ ]:
# -- CELL 2: Kiểm tra GPU ----------------------------------
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  KHÔNG CÓ GPU! Vào Settings → Accelerator → GPU T4')

In [ ]:
# -- CELL 3: Thiết lập đường dẫn (AUTO-DETECT) -------------
import os, sys, zipfile, shutil, glob

WORK_CODE = '/kaggle/working/code'
os.makedirs(WORK_CODE, exist_ok=True)

# --- TÌM odir5k-code dataset (có thể nằm ở nhiều nơi) ---
CODE_INPUT = None
for candidate in [
    '/kaggle/input/odir5k-code',
    '/kaggle/input/datasets/ngodinhdatcpp/odir5k-code',
    '/kaggle/input/datasets/odir5k-code',
]:
    if os.path.isdir(candidate):
        CODE_INPUT = candidate
        break

# Fallback: tìm bất kỳ thư mục nào chứa train.py
if not CODE_INPUT:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.py' in files:
            CODE_INPUT = root
            break

if CODE_INPUT:
    print(f'✅ Tìm thấy code tại: {CODE_INPUT}')
    print(f'   Nội dung: {os.listdir(CODE_INPUT)}')
else:
    print('❌ KHÔNG tìm thấy dataset odir5k-code!')
    print('   Hãy Add Input → search "odir5k-code"')
    raise FileNotFoundError('odir5k-code not found')

# --- COPY / GIẢI NÉN code vào working ---
for name in ['src', 'configs', 'splits_clean']:
    dest = f'{WORK_CODE}/{name}'
    if os.path.exists(dest):
        print(f'✅ {name}/ (đã có)')
        continue
    dir_path = f'{CODE_INPUT}/{name}'
    zip_path = f'{CODE_INPUT}/{name}.zip'
    if os.path.isdir(dir_path):
        shutil.copytree(dir_path, dest)
        print(f'✅ {name}/ (copy từ dataset)')
    elif os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(WORK_CODE)
        print(f'✅ {name}.zip → giải nén')
    else:
        print(f'❌ Không tìm thấy {name}')

for fname in ['train.py', 'evaluate.py']:
    src = f'{CODE_INPUT}/{fname}'
    dst = f'{WORK_CODE}/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'✅ {fname}')

sys.path.insert(0, WORK_CODE)
CODE_DIR   = WORK_CODE
SPLITS_DIR = f'{WORK_CODE}/splits_clean'

# --- TÌM ẢNH ODIR-5K (Training Images) ---
RAW_DIR = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'Training Images' in dirs:
        candidate = os.path.join(root, 'Training Images')
        n = len([f for f in os.listdir(candidate) if f.endswith('.jpg')])
        if n > 1000:  # Đúng ODIR-5K phải có >6000 ảnh
            RAW_DIR = candidate
            break

if RAW_DIR:
    n = len([f for f in os.listdir(RAW_DIR) if f.endswith('.jpg')])
    print(f'\n✅ RAW_DIR: {RAW_DIR} ({n} ảnh)')
else:
    print('\n❌ KHÔNG tìm thấy Training Images!')
    print('   → Add Input → tab Datasets → search "ocular disease recognition"')
    RAW_DIR = '/kaggle/input/FIX_ME'

ROI_DIR = '/kaggle/working/preprocessed_images'
ENH_DIR = '/kaggle/working/enhanced_images'
RES_DIR = '/kaggle/working/results'
for d in [ROI_DIR, ENH_DIR, RES_DIR]:
    os.makedirs(d, exist_ok=True)

# --- KIỂM TRA IMPORT ---
try:
    from src.models import build_model
    from src.loss import MultiTaskLoss
    print(f'✅ import src.* OK')
except Exception as e:
    print(f'❌ Import lỗi: {e}')

print(f'\n📋 TỔNG KẾT:')
print(f'  CODE_DIR   = {CODE_DIR}')
print(f'  SPLITS_DIR = {SPLITS_DIR}')
print(f'  RAW_DIR    = {RAW_DIR}')
print(f'  ROI_DIR    = {ROI_DIR}')
print(f'  ENH_DIR    = {ENH_DIR}')


In [ ]:
# -- CELL 5: ROI Crop --------------------------------------
import cv2, numpy as np
from pathlib import Path
from tqdm import tqdm

def crop_roi(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    rows, cols = np.where(mask)
    if len(rows) == 0: return cv2.resize(img, (512,512))
    r0,r1,c0,c1 = rows.min(),rows.max(),cols.min(),cols.max()
    return cv2.resize(img[r0:r1+1, c0:c1+1], (512,512))

imgs = list(Path(RAW_DIR).glob('*.jpg'))
print(f'Bắt đầu ROI Crop {len(imgs)} ảnh...')
for p in tqdm(imgs, desc='ROI Crop'):
    img = cv2.imread(str(p))
    if img is None: continue
    cv2.imwrite(str(Path(ROI_DIR)/p.name), crop_roi(img))
print(f'✅ ROI Crop xong: {len(list(Path(ROI_DIR).glob("*.jpg")))} ảnh')

In [ ]:
# -- CELL 6: Ben Graham + CLAHE ----------------------------
def ben_graham(img, sigma_ratio=1/6, scale=128):
    h,w = img.shape[:2]
    sigma = int(max(h,w)*sigma_ratio)
    if sigma%2==0: sigma+=1
    local = cv2.GaussianBlur(img.astype(np.float32),(0,0),sigmaX=sigma)
    return np.clip(img.astype(np.float32)-local+scale,0,255).astype(np.uint8)

def apply_clahe(img):
    lab = cv2.cvtColor(img,cv2.COLOR_BGR2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    return cv2.cvtColor(cv2.merge([clahe.apply(l),a,b]),cv2.COLOR_LAB2BGR)

roi_imgs = list(Path(ROI_DIR).glob('*.jpg'))
print(f'Ben Graham + CLAHE trên {len(roi_imgs)} ảnh...')
for p in tqdm(roi_imgs, desc='Enhance'):
    img = cv2.imread(str(p))
    if img is None: continue
    cv2.imwrite(str(Path(ENH_DIR)/p.name), apply_clahe(ben_graham(img)))
print(f'✅ Enhanced xong: {len(list(Path(ENH_DIR).glob("*.jpg")))} ảnh')

In [ ]:
# -- CELL 7: Load metadata & pos_weight -------------------
import json
meta     = json.load(open(f'{SPLITS_DIR}/metadata.json'))
age_mean = meta['age_stats']['mean']
age_std  = meta['age_stats']['std']
LABELS   = meta['labels']
pos_weight = torch.FloatTensor([meta['class_weights'][l] for l in LABELS])

print(f'Labels: {LABELS}')
print(f'age_mean={age_mean:.2f}, age_std={age_std:.2f}')
print('pos_weight:', {l:round(v,2) for l,v in zip(LABELS,pos_weight.tolist())})

In [ ]:
    # -- CELL 8: Hàm training đầy đủ (với WeightedRandomSampler) ------
    import yaml, time, random
    import numpy as np
    import pandas as pd
    from torch.utils.data import DataLoader, WeightedRandomSampler
    from sklearn.metrics import f1_score, roc_auc_score
    from src.dataset import ODIRDataset
    from src.transforms import get_transforms
    from src.models import build_model
    from src.loss import MultiTaskLoss
    from src.mixup import MixUpCollator
    from src.cutmix import CutMixCollator
    from src.utils import LABELS as SRC_LABELS

    def make_weighted_sampler(ds):
        """Tính WeightedRandomSampler cân bằng lớp cho multi-label dataset."""
        labels_matrix = ds.df[SRC_LABELS].values.astype(float)
        class_counts  = labels_matrix.sum(axis=0)
        class_weights = 1.0 / np.maximum(class_counts, 1.0)
        sample_weights = np.dot(labels_matrix, class_weights)
        sample_weights = np.maximum(sample_weights, 1e-5)
        print(f'[WRS] Trọng số lớp: {dict(zip(SRC_LABELS, np.round(class_weights,4)))}')
        return WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )

    def run_experiment(config_path, img_dir_override=None):
        """Chạy 1 thực nghiệm từ file YAML config."""
        cfg = yaml.safe_load(open(config_path))
        exp_name   = cfg['experiment_name']
        model_type = cfg.get('model_type', 'swin')
        tr_cfg     = cfg['training']
        aug_cfg    = cfg.get('augmentation', {})
        img_size   = tr_cfg['img_size']
        batch      = tr_cfg['batch_size']
        epochs     = tr_cfg['epochs']
        patience   = tr_cfg.get('early_stopping_patience', 8)
        use_wrs    = tr_cfg.get('use_weighted_sampler', False)

        if img_dir_override:
            img_dir = img_dir_override
        elif 'enhanced' in cfg.get('img_dir', ''):
            img_dir = ENH_DIR
        else:
            img_dir = RAW_DIR

        out_dir = f'{RES_DIR}/{exp_name}'
        os.makedirs(out_dir, exist_ok=True)
        print(f"\n{'='*55}\n  {exp_name}\n  img_dir={img_dir}\n{'='*55}")

        tf_train = get_transforms('train', img_size)
        tf_val   = get_transforms('val',   img_size)

        def mk_ds(split, tf):
            return ODIRDataset(f'{SPLITS_DIR}/{split}.csv', img_dir, tf, age_mean, age_std)

        ds_train = mk_ds('train', tf_train)
        ds_val   = mk_ds('val',   tf_val)
        ds_test  = mk_ds('test',  tf_val)

        # -- Train loader: WeightedRandomSampler + MixUp/CutMix --
        use_mixup  = aug_cfg.get('use_mixup', False)
        use_cutmix = aug_cfg.get('use_cutmix', False)

        if use_wrs:
            sampler = make_weighted_sampler(ds_train)
            base_kw = dict(batch_size=batch, sampler=sampler,
                           num_workers=2, pin_memory=True, drop_last=True)
        else:
            base_kw = dict(batch_size=batch, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)

        class _MixCut:
            def __init__(self):
                self.mx = MixUpCollator(alpha=aug_cfg.get('mixup_alpha',0.4), prob=1.0)
                self.cx = CutMixCollator(alpha=aug_cfg.get('cutmix_alpha',1.0), prob=1.0)
            def __call__(self, b): return self.mx(b) if random.random()<0.5 else self.cx(b)

        if use_mixup and use_cutmix:
            train_loader = DataLoader(ds_train, **base_kw, collate_fn=_MixCut())
            print('[Aug] MixUp + CutMix xen kẽ')
        elif use_mixup:
            train_loader = DataLoader(ds_train, **base_kw,
                                      collate_fn=MixUpCollator(alpha=aug_cfg.get('mixup_alpha',0.4), prob=0.5))
            print('[Aug] MixUp only')
        elif use_cutmix:
            train_loader = DataLoader(ds_train, **base_kw,
                                      collate_fn=CutMixCollator(alpha=aug_cfg.get('cutmix_alpha',1.0), prob=0.5))
            print('[Aug] CutMix only')
        else:
            train_loader = DataLoader(ds_train, **base_kw)
            print('[Aug] Không dùng MixUp/CutMix')

        val_loader  = DataLoader(ds_val,  batch_size=batch, num_workers=2)
        test_loader = DataLoader(ds_test, batch_size=batch, num_workers=2)

        model = build_model(
            model_type=model_type, pretrained=True,
            img_size=img_size,
            variant=cfg.get('model', {}).get('variant', 'tiny')
        ).to(device)
        pw        = pos_weight.to(device)
        criterion = MultiTaskLoss(pos_weight=pw, lam=cfg['loss'].get('lam', 0.1))
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg['optimizer'].get('lr', 3e-4),
            weight_decay=cfg['optimizer'].get('weight_decay', 0.05)
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=epochs, eta_min=1e-6
        )

        def run_epoch(loader, mode, threshold=0.5):
            model.train() if mode == 'train' else model.eval()
            tot = 0; probs = []; tgts = []; ap = []; at = []
            ctx = torch.enable_grad() if mode == 'train' else torch.no_grad()
            with ctx:
                for batch_data in loader:
                    imgs = batch_data['image'].to(device)
                    lbl  = batch_data['labels'].to(device)
                    age  = batch_data['age'].to(device)
                    out  = model(imgs)
                    loss, _ = criterion(out['logits'], lbl, out['age_pred'], age)
                    if mode == 'train':
                        optimizer.zero_grad(); loss.backward()
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                        optimizer.step()
                    tot += loss.item() * imgs.size(0)
                    probs.extend(torch.sigmoid(out['logits']).detach().cpu().tolist())
                    tgts.extend(lbl.detach().cpu().tolist())
                    ap.extend(out['age_pred'].squeeze(1).detach().cpu().tolist())
                    at.extend(age.squeeze(1).detach().cpu().tolist())
            p = np.array(probs); t = np.array(tgts)
            if t.ndim > 1 and not np.all(np.isin(t, [0, 1])):
                t = (t >= 0.5).astype(int)
            thresh_np = np.array(threshold) if isinstance(threshold, list) else np.array([threshold]*p.shape[1])
            f1  = f1_score(t, (p >= thresh_np).astype(int), average='macro', zero_division=0)
            try:  auc = roc_auc_score(t, p, average='macro')
            except: auc = -1.0
            ap2 = np.array(ap)*age_std + age_mean
            at2 = np.array(at)*age_std + age_mean
            mae = float(np.mean(np.abs(ap2 - at2)))
            return {'loss': tot/len(loader.dataset), 'f1': f1, 'auc': auc, 'mae': mae,
                    'probs': probs, 'targets': tgts}

        best_f1 = 0; wait = 0; best_state = None; log = []
        for ep in range(1, epochs+1):
            t0 = time.time()
            tm = run_epoch(train_loader, 'train')
            vm = run_epoch(val_loader,   'val')
            scheduler.step()
            print(f'Ep {ep:02d}/{epochs} | Train={tm["loss"]:.4f} | '
                  f'Val F1={vm["f1"]:.4f} AUC={vm["auc"]:.4f} MAE={vm["mae"]:.1f}y '
                  f'[{time.time()-t0:.1f}s]')
            log.append({'ep': ep, 'train_loss': tm['loss'], 'val_f1': vm['f1'],
                        'val_auc': vm['auc'], 'val_mae': vm['mae']})
            if vm['f1'] > best_f1:
                best_f1 = vm['f1']; wait = 0
                best_state = model.state_dict()
                torch.save(best_state, f'{out_dir}/best.pth')
            else:
                wait += 1
                if wait >= patience:
                    print(f'  \u23f9 Early stop at ep {ep}'); break

        model.load_state_dict(best_state)

        print('[Dynamic Thresholding] Đang quét tìm ngưỡng tối ưu trên tập Validation...')
        val_preds = run_epoch(val_loader, 'val')
        from src.utils import find_best_thresholds, get_label_names
        import torch as _torch
        best_thresholds = find_best_thresholds(
            _torch.FloatTensor(val_preds['probs']),
            _torch.FloatTensor(val_preds['targets'])
        )
        print('  \u2192 Ngưỡng tối ưu tìm được cho 8 bệnh lý:')
        lbl_names = get_label_names()
        for idx, lbl in enumerate(LABELS):
            print(f'    - {lbl} ({lbl_names[lbl]}): {best_thresholds[idx]:.2f}')
        print()

        test_m_default = run_epoch(test_loader, 'test', threshold=0.5)
        print(f'\nTEST (Ngưỡng mặc định 0.5): F1={test_m_default["f1"]:.4f} '
              f'AUC={test_m_default["auc"]:.4f} MAE={test_m_default["mae"]:.1f}y')

        test_m_opt = run_epoch(test_loader, 'test', threshold=best_thresholds)
        print(f'TEST (Ngưỡng tối ưu động):  F1={test_m_opt["f1"]:.4f} '
              f'AUC={test_m_opt["auc"]:.4f} MAE={test_m_opt["mae"]:.1f}y')

        result = {
            'exp': exp_name,
            'best_val_f1_default': best_f1,
            'optimal_thresholds': best_thresholds,
            'test_default_0.5': test_m_default,
            'test_optimal_dynamic': test_m_opt,
            'log': log
        }
        import json as _json2
        _json2.dump(result, open(f'{out_dir}/results.json', 'w'), indent=2, default=str)
        return result


In [ ]:
# -- CELL 9: EXP 4 — Swin + Ảnh gốc ---------------------
r1 = run_experiment(
    config_path = f'{CODE_DIR}/configs/exp_4_swin_no_preprocess.yaml',
    img_dir_override = RAW_DIR,
)


In [ ]:
# -- CELL 10: EXP 5 — Swin + Enhanced --------------------
r2 = run_experiment(
    config_path = f'{CODE_DIR}/configs/exp_5_swin_preprocess_no_aug.yaml',
    img_dir_override = ENH_DIR,
)


In [ ]:
# -- CELL 11: EXP 6 — Swin + Enhanced + Aug --------------
r3 = run_experiment(
    config_path = f'{CODE_DIR}/configs/exp_6_swin_preprocess_with_aug.yaml',
    img_dir_override = ENH_DIR,
)


In [ ]:
# -- CELL 12: Bảng so sánh kết quả -------------------------
import pandas as pd

exp_labels = {
    'exp_4_swin_no_preprocess': 'EXP 4: Raw (không xử lý)',
    'exp_5_swin_preprocess_no_aug': 'EXP 5: Enhanced (không aug)',
    'exp_6_swin_preprocess_with_aug': 'EXP 6: Enhanced + MixUp+CutMix',
}

rows = []
for r in [r1,r2,r3]:
    # Dùng key mới từ Dynamic Thresholding
    t     = r.get('test_optimal_dynamic', r.get('test_default_0.5', {}))
    t_def = r.get('test_default_0.5', {})
    best_val = r.get('best_val_f1_default', r.get('best_val_f1', 0))
    rows.append({
        'Thực nghiệm'           : exp_labels.get(r['exp'], r['exp']),
        'Best Val F1'            : round(best_val, 4),
        'Test F1 (ngưỡng 0.5)'  : round(t_def.get('f1', 0), 4),
        'Test F1 (ngưỡng tối ưu)': round(t.get('f1', 0), 4),
        'Test AUC-ROC'           : round(t.get('auc', 0), 4),
        'Test Age MAE'           : round(t.get('mae', 0), 2),
    })

df = pd.DataFrame(rows).set_index('Thực nghiệm')
print(df.to_string())

# Phân tích delta
def _f1(r): return r.get('test_optimal_dynamic', r.get('test_default_0.5', {})).get('f1', 0)
d12 = _f1(r2) - _f1(r1)
d23 = _f1(r3) - _f1(r2)
d13 = _f1(r3) - _f1(r1)
print(f'\n[PHÂN TÍCH]')
print(f'Tiền xử lý (EXP5-EXP4):    {d12:+.4f} → {"✅ Có ích" if d12>0.01 else "≈ Không đáng kể"}')
print(f'MixUp+CutMix (EXP6-EXP5):  {d23:+.4f} → {"✅ Có ích" if d23>0.005 else "≈ Không đáng kể"}')
print(f'Tổng cải thiện (EXP6-EXP4): {d13:+.4f}')

# Lưu bảng markdown
md_lines = ['| Thực nghiệm | Test F1 | Test AUC | Test MAE |','|---|---|---|---|']
for _, row in df.iterrows():
    md_lines.append(f"| {_.strip()} | {row['Test F1 (ngưỡng tối ưu)']} | {row['Test AUC-ROC']} | {row['Test Age MAE']} |")
open(f'{RES_DIR}/comparison_table.md','w').write('\n'.join(md_lines))
print(f'\n✅ Bảng so sánh lưu tại: {RES_DIR}/comparison_table.md')


In [ ]:
# -- CELL 13: Vẽ Learning Curves Swin --------------------
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1,2,figsize=(14,5))
colors = ['#e74c3c','#3498db','#2ecc71']
exp_names = ['EXP4: Swin Raw','EXP5: Swin Enhanced','EXP6: Swin Enhanced+Aug']

for r,c,lbl in zip([r1,r2,r3], colors, exp_names):
    eps = [x['ep'] for x in r['log']]
    f1s = [x['val_f1'] for x in r['log']]
    maes = [x['val_mae'] for x in r['log']]
    axes[0].plot(eps,f1s,c=c,label=lbl,linewidth=2)
    axes[1].plot(eps,maes,c=c,label=lbl,linewidth=2)

axes[0].set(title='Swin — Val F1-macro',xlabel='Epoch',ylabel='F1-macro')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set(title='Swin — Val Age MAE',xlabel='Epoch',ylabel='MAE (years)')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/swin_learning_curves.png', dpi=150)
plt.show()


In [ ]:
# -- CELL 14: Confusion Analysis --------------------------
# Chạy confusion_analysis.py cho SWIN để tìm ra bệnh lý dễ bị nhầm lẫn nhất
import os
print('Running Confusion Analysis for SWIN...')
cmd = f'PYTHONPATH={CODE_DIR} python3 {CODE_DIR}/confusion_analysis.py --model swin --checkpoint {RES_DIR}/exp_6_swin_preprocess_with_aug/best.pth --img_dir {ENH_DIR} --splits_dir {SPLITS_DIR} --output {RES_DIR}/confusion_analysis_swin.md'
print(f'Executing command: {cmd}')
os.system(cmd)

# Hiển thị báo cáo tóm tắt
report_path = f'{RES_DIR}/confusion_analysis_swin.md'
if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print(f'❌ Không tìm thấy báo cáo: {report_path}')


In [ ]:
# -- CELL 15: Ensemble Evaluation (CNN + Swin Transformer) --
# Lưu ý: Cần có cả 2 checkpoint để chạy:
#   1. CNN: results/exp_3_cnn_preprocess_with_aug/best.pth
#   2. SWIN: results/exp_6_swin_preprocess_with_aug/best.pth
import os
cnn_ckpt = f'{RES_DIR}/exp_3_cnn_preprocess_with_aug/best.pth'
swin_ckpt = f'{RES_DIR}/exp_6_swin_preprocess_with_aug/best.pth'

if os.path.exists(cnn_ckpt) and os.path.exists(swin_ckpt):
    print('✅ Tìm thấy đầy đủ checkpoint. Bắt đầu đánh giá Ensemble...')
    cmd = f'PYTHONPATH={CODE_DIR} python3 {CODE_DIR}/ensemble_evaluate.py --cnn_checkpoint {cnn_ckpt} --swin_checkpoint {swin_ckpt} --img_dir {ENH_DIR} --splits_dir {SPLITS_DIR} --output_dir {RES_DIR}'
    print(f'Executing command: {cmd}')
    os.system(cmd)
    
    # Hiển thị báo cáo kết quả
    report_path = f'{CODE_DIR}/docs/ensemble_report.md'
    if os.path.exists(report_path):
        with open(report_path, 'r', encoding='utf-8') as f:
            print(f.read())
    else:
        print(f'❌ Không tìm thấy báo cáo ensemble: {report_path}')
else:
    print('⚠️ Thiếu ít nhất 1 checkpoint để chạy Ensemble!')
    print(f'  - Checkpoint CNN: [{"✅ Đã tìm thấy" if os.path.exists(cnn_ckpt) else "❌ Không tìm thấy"}] tại {cnn_ckpt}')
    print(f'  - Checkpoint Swin: [{"✅ Đã tìm thấy" if os.path.exists(swin_ckpt) else "❌ Không tìm thấy"}] tại {swin_ckpt}')
    print('  Mẹo: Bạn có thể copy checkpoint từ các notebook run khác hoặc Add Output của notebook khác vào làm Input.')
